In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
from io import BytesIO

In [ ]:
def step(queue, is_used, lattice):
  L = lattice.shape[0]
  a_idx, b_idx = queue.popleft()
  is_used[a_idx, b_idx] = True

  dirs = [
    (0, -1),   # left
    (0,  1),   # right
    (1,  0),   # top
    (-1, 0),   # bottom
  ]

  neighbours = []
  
  for da, db in dirs:
    na, nb = a_idx + da, (b_idx + db) % L

    if 0 <= na < L:
      pos = (na, nb)

      if not is_used[pos]:
        is_used[pos] = True

        if lattice[pos]:
          neighbours.append(pos)

  queue.extend(neighbours)

In [ ]:
L = 20
p = 0.61

is_used = np.array([[i==0 for _ in range(L)] for i in range(L)])
lattice = np.random.rand(L, L) < p
queue = deque([[0, i] for i in range(L)])

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
ax.axis("off")

with imageio.get_writer('../results/gifs/07_perkolation.gif', mode='I', duration=0.02) as writer:
  while queue: # BFS
    ax.clear()
    step(queue, is_used, lattice)

    frame = [[
      2 if is_used[i, j] and lattice[i, j] else
      1 if is_used[i, j] and not lattice[i, j] else
      0 for j in range(L)] for i in range(L)]

    cmap = ListedColormap(['indigo', 'darkolivegreen', 'gold'])
    plt.imshow(frame, cmap=cmap, interpolation='nearest')
    
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    writer.append_data(imageio.imread(buf))

In [ ]:
def perkolacje(L, p, N):
  perkolacje_count = 0
  size_count, cluster_count = 0, 0
  
  for _ in range(N):
    is_used = np.array([[i==0 for _ in range(L)] for i in range(L)])
    lattice = np.random.rand(L, L) < p
    queue = deque([[0, i] for i in range(L)])
    
    while queue:
      step(queue, is_used, lattice)

    if np.any(is_used[-1] & lattice[-1]):
      perkolacje_count += 1
    
    for i, j in np.argwhere(~is_used):
      is_used[i, j] = True

      if lattice[i, j]:
        cluster_count += 1
        queue.append([i, j])

        while queue:
          size_count += 1
          step(queue, is_used, lattice)

  return [(size_count / cluster_count) if cluster_count else 0, (perkolacje_count / N)]

In [ ]:
L20, L50, L100 = [], [], []
p_vals = np.linspace(0.4, 0.70, 60)
N = 100

L20  = np.array([perkolacje(20, p, N) for p in p_vals])
L50  = np.array([perkolacje(50, p, N) for p in p_vals])
L100 = np.array([perkolacje(100, p, N) for p in p_vals])


fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(p_vals, L20[:, 1], 'o-', label='L=20 P(p)')
ax.plot(p_vals, L50[:, 1], 's-', label='L=50 P(p)')
ax.plot(p_vals, L100[:, 1], '^-', label='L=100 P(p)')

ax.set_title("Percolation P(p)")
ax.set_xlabel("p")
ax.set_ylabel("P(p)")
ax.legend()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(p_vals, L20[:, 0], 'o-', label='L=20 S(p)')
ax.plot(p_vals, L50[:, 0], 's-', label='L=50 S(p)')
ax.plot(p_vals, L100[:, 0], '^-', label='L=100 S(p)')

ax.set_title("Percolation S(p)")
ax.set_xlabel("p")
ax.set_ylabel("S(p)")
ax.legend()
plt.show()
plt.close(fig)

# approx 5 minutes